# 7.04 Playwright Browser Toolkit — 에이전트에게 실제 브라우저를 주기

`PlayWrightBrowserToolkit` 은 **Chromium/Firefox/WebKit** 을 자동 조종하는 7개 도구를 에이전트에게 묶어 준다. 일반 HTTP 스크랩으로는 닿지 못하는 **JavaScript 렌더링 페이지·로그인 세션·양식 제출**을 다룰 수 있다.

도구 목록:
- `navigate_browser(url)` — 페이지 이동
- `previous_webpage()` — 뒤로 가기
- `click_element(selector)` — CSS selector 클릭
- `extract_text()` — 현재 페이지 텍스트 추출
- `extract_hyperlinks()` — 링크 목록
- `get_elements(selector, attributes=...)` — selector 로 엘리먼트 리스트
- `current_webpage()` — 현재 URL · 제목

## 학습 목표

- `playwright install chromium` 으로 브라우저 바이너리 다운로드
- 비동기 (`async_browser`) vs 동기 (`sync_browser`) 생성 방식 차이
- 에이전트가 `navigate` → `extract_text` 순서로 페이지 내용을 읽어오는 표준 플로우
- 양식 제출·로그인 자동화에는 반드시 **HITL** 을 끼우는 패턴

## 언제 쓰나

- SPA(React/Vue) 로 만들어진 사이트 — `requests` 로는 빈 HTML
- 로그인 뒤에만 보이는 대시보드 데이터 수집
- 검색 결과 필터를 **클릭**해야만 나오는 리스트

## 7.04.1 환경 설정

`playwright` Python 패키지 + `playwright install chromium` (브라우저 바이너리, ~150MB). 둘 다 있어야 실행된다. probe 는 실제 `p.chromium.launch()` → `new_page()` 로 확인.

In [ ]:
# !pip install -U langchain langchain-community playwright beautifulsoup4 lxml
# !playwright install chromium

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# probe: chromium 바이너리와 playwright 런타임이 실제로 동작하는지 확인 (Jupyter top-level await 사용)
from playwright.async_api import async_playwright

async def _probe():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto("https://example.com")
        title = await page.title()
        await browser.close()
        return title

probe_title = await _probe()
print("probe OK, page title =", probe_title)

## 7.04.2 기본 사용 — Toolkit 초기화 + 7개 도구 탐색

`PlayWrightBrowserToolkit.from_browser(async_browser=...)` 로 toolkit 을 만든다. 도구들은 대부분 async — `.ainvoke(...)` 로 호출한다.

In [ ]:
from langchain_community.agent_toolkits.playwright.toolkit import PlayWrightBrowserToolkit

# 노트북 세션 내내 재사용할 브라우저 — 마지막 셀에서 close
playwright_ctx = await async_playwright().start()
browser = await playwright_ctx.chromium.launch(headless=True)

toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=browser)
pw_tools = toolkit.get_tools()
for t in pw_tools:
    print("-", t.name, "|", (t.description or "").split("\n")[0][:90])

## 7.04.3 수동 시퀀스 — `navigate` → `extract_text`

도구 직접 호출로 플로우를 따라가 본다. **프로덕션에선 에이전트에게 맡기지만, 디버그 시 수동 호출이 유용.**

In [ ]:
tool_by_name = {t.name: t for t in pw_tools}

nav = tool_by_name["navigate_browser"]
extract = tool_by_name["extract_text"]
current = tool_by_name["current_webpage"]

print(await nav.ainvoke({"url": "https://example.com"}))
print("현재:", await current.ainvoke({}))
text = await extract.ainvoke({})
print("본문 일부:", text[:200])

## 7.04.4 `create_agent` 에 연결 — 웹 스크래핑 에이전트

에이전트가 스스로 **`navigate` → 내용 확인 → 필요하면 클릭** 시퀀스를 구성하게 한다. 모델은 도구 시그니처(selector/url 인자) 를 보고 어떤 순서로 호출할지 결정.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain.agents import create_agent

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=pw_tools,
        system_prompt=(
            "브라우저 도구로 주어진 URL 을 탐색해 요청된 정보를 추출한다. "
            "절대 로그인/결제/양식 제출 시도 금지 — 읽기 전용."
        ),
    )
    res = await agent.ainvoke({
        "messages": [{"role": "user", "content": "https://example.com 에 접속해서 페이지 제목과 첫 단락을 알려줘."}]
    })
    print(res["messages"][-1].content[:400])
else:
    print("OPENAI_API_KEY 없음 → 에이전트 예제 스킵. 도구 동작은 7.04.3 에서 확인됨.")

## 7.04.5 주요 옵션 · 비교 표

### 브라우저 생성 방식

| 방식 | API | 장점 | 주의 |
|------|-----|------|------|
| Async | `async_playwright()` + `await p.chromium.launch()` | Jupyter/LangGraph 에이전트와 자연스럽게 통합 | `ainvoke` 사용 필수 |
| Sync  | `sync_playwright().start().chromium.launch()` | 일반 스크립트·CLI | Jupyter 의 running loop 와 충돌 가능 |
| community helper | `create_async_playwright_browser()` | 한 줄로 launch | 내부가 nested event loop 호출 — **plain Python 에서 RuntimeError** 유발. Jupyter에서도 주의 |

### launch 옵션

| 옵션 | 기본 | 용도 |
|------|------|------|
| `headless` | True | 눈에 안 보이게. 디버그 시 False |
| `slow_mo` | 0 | 각 액션 사이 지연(ms) — 데모용 |
| `proxy` | None | 프록시 경유 필요 시 |
| `user_agent` | 자동 | 일부 사이트가 Playwright UA 차단 — 커스텀 UA 로 우회 |

### 셀렉터 전략

- CSS 가 기본. `click_element(selector="button.primary")`
- 텍스트 기반은 `get_elements(selector="text=로그인")` (Playwright 확장 셀렉터)
- 프레임 내부 요소는 기본 도구로 접근 제한 — 커스텀 tool 작성 필요

## 7.04.6 (필수) HITL 결합 — 쓰기 액션 승인

브라우저 자동화는 **실제 원격 서비스에 영향을 준다**. 쿠키·로그인·양식 제출이 걸리면 HITL 없이 에이전트를 돌리면 안 된다.

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware

# 읽기 도구 whitelist
READ_ONLY = {"navigate_browser", "previous_webpage", "extract_text",
             "extract_hyperlinks", "get_elements", "current_webpage"}

write_tools = [t for t in pw_tools if t.name not in READ_ONLY]

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=pw_tools,
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            t.name: {"allowed_decisions": ["approve", "edit", "reject"]}
            for t in write_tools
        },
    )],
)
```

`click_element` 같은 도구는 쇼핑몰 구매 버튼·삭제 버튼에도 쓰일 수 있어 반드시 승인 게이트로 감싼다.

## 7.04.7 정리 — 브라우저 close

리소스 누수 방지. 노트북 종료 전 명시적 close.

In [ ]:
await browser.close()
await playwright_ctx.stop()
print("브라우저 종료 완료")

## 체크리스트

- [ ] `playwright install chromium` 은 **브라우저 바이너리** 를 받는 단계 — 패키지 설치와 별개
- [ ] 에이전트에는 기본 **읽기 전용 도구**만 노출, 쓰기 도구(`click_element`)는 HITL
- [ ] 세션 쿠키가 필요한 사이트는 `browser.new_context(storage_state="auth.json")` 로 로그인 재사용
- [ ] 대량 페이지 탐색엔 **각 방문 사이 delay** (`slow_mo`) 로 사이트에 부담 줄이기
- [ ] Jupyter 가 아닌 순수 스크립트에서는 `async_playwright()` 를 `asyncio.run(...)` 안에서 호출
- [ ] 크롤링 대상의 **robots.txt · ToS** 확인 — 법적 위험 관리

## 다음

- `07_integration/04_document_loaders/02_web_crawl.ipynb` — 정적 HTML 로더 (WebBaseLoader, SitemapLoader) — JS 렌더 불필요한 페이지에 더 빠름
- `02_langchain/07_hitl_and_runtime` — `click_element` 같은 쓰기 도구의 HITL 전체 resume 흐름

## 참고

- Playwright Python: https://playwright.dev/python/
- LangChain Playwright Toolkit: https://python.langchain.com/docs/integrations/tools/playwright/